# Corpus & Vocabulary Embedding + Scoring Run

Standalone embed + score notebook for the **30 Aug 2026 retraining models**.
It does **not** train anything — it consumes the staged products of the training
pipeline run folders and writes one self-contained scoring run folder.

## Run matrix

| Model (internal code) | Trained on | Scores with | Scored on corpuses | Scored on vocabs |
|---|---|---|---|---|
| A1 (cosine finetune) | legacy corpus | legacydict | legacy + policy | legacy + policy |
| A2 (BERTopic finetune) | legacy corpus | legacydict | legacy + policy | legacy + policy |
| B1 (cosine finetune) | policy corpus | policydict | legacy + policy | legacy + policy |
| pre (pretrained e5 baseline) | — | legacydict **and** policydict | legacy + policy | legacy + policy |

Ground rules baked into this notebook:

- **No cross-dictionary combinations.** A finetuned model is only scored against the
  dictionary it learned from (A1/A2 → legacydict, B1 → policydict). Scoring a model on
  the *other corpus* is fine and intended; scoring it with the other *dictionary* is not.
- **Mean-centering is per space** (space = modality × corpus × model). Chunk scoring
  centres chunks and seed centroids with the chunk-space mean; vocab scoring centres
  terms and seed centroids with the term-space mean. Raw (uncentred) embeddings are what
  is saved to disk; the means are saved separately so any consumer can re-centre.
- **All claims are within-corpus.** Legacy-corpus scores and policy-corpus scores are
  never comparable in absolute terms (corpus-specific centering).
- **Stable chunk identity**: every chunk row carries `content_key` = SHA-1 of the
  normalized chunk text, stored alongside `doc_id` and `chunk_index`, so old and new
  scores can be matched even if chunk boundaries shift.
- **Isolated-string encodings only** for vocab and seed terms — never mix these
  products with the corpus-contextual `Viz_terms` expansion embeddings.
- Model codes A1/A2/B1 are **internal labels** (filenames, configs, logs). Reader-facing
  text uses pretrained / legacy-trained / policy-trained.

> **A1 is still training?** No problem — the discovery cell skips any model whose
> `trained_encoder` folder is missing and scores the rest. Re-run the notebook once A1
> lands; with `RESUME = True` finished products are not recomputed.

## Output layout

```
<TRAIN_WORKFLOW_DIR>/full_embed_score_e5_<YYYYMMDD>/
├── config/run_config_<ts>.json          # resolved paths, counts, settings snapshot
├── embeddings/
│   ├── chunks/<corpus>/<model>.npy      # raw float32, row order == index.parquet
│   ├── chunks/<corpus>/index.parquet    # content_key, doc_id, chunk_index
│   ├── vocab/<corpus>/<model>.npy       # raw float32, row order == index.parquet
│   ├── vocab/<corpus>/index.parquet     # term, term_type, frequency
│   ├── seeds/<dict>/<model>.npy         # seed-term embeddings (raw)
│   └── means/<space>__<corpus>__<model>.npy
├── Chunk_scores/chunk_scores_<corpus>__dict-<dict>__model-<model>.parquet
├── Chunk_scores/chunk_scores_combined_<corpus>.parquet   (+ .xlsx for small corpora)
├── Vocab_scores/vocab_scores_<corpus>__dict-<dict>__model-<model>.parquet
└── reports/score_spread_summary.csv
```

The **combined table** is one wide row-per-chunk table per corpus with fixed leading
columns — `chunk_index, Raw_text, Doc_type, Year, notes, Region` — followed by one
score block per model in the order **pretrained, A1, A2, B1** (`doc_id` and
`content_key` close the table for traceability). The `notes` column flags chunks
found in the configured `NOTES_SOURCES` files (e.g. *used in prose*, *in previous
dataset*), matched on the stable content key.


In [ ]:
# ============================================================
# CELL 1: IMPORTS & DEVICE
# ============================================================
import os
import re
import json
import hashlib
import unicodedata
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | sentence-transformers ready | device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: no GPU detected - embedding the policy corpus on CPU will be slow.")

---
## Configuration

Edit **only** this cell. Everything below resolves paths, validates products and runs.


In [ ]:
# ============================================================
# CELL 2: CONFIGURATION  (edit this cell only)
# ============================================================

# --- project root ---------------------------------------------------------
PROJECT_ROOT = Path(os.environ.get("POLICY_ANALYSIS_ROOT", r"C:\Users\Home\policy-analysis"))
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

# --- the workflow folder the retraining run writes into -------------------
# Iteration discipline: every pipeline run gets its own workflow_NN under the
# project root. Point this at the retraining run's folder.
TRAIN_WORKFLOW_DIR = PROJECT_ROOT / "workflow_45"    # <-- EDIT if the run landed elsewhere

# --- training-pipeline run folders ----------------------------------------
# Each is a run folder with the standard layout (Other_data/, Dictionary/,
# Model_finetuning/trained_encoder/, ...). Leave None to let the discovery
# cell list candidates found under TRAIN_WORKFLOW_DIR, then paste the right
# one here (as str or Path).
RUN_FOLDERS = {
    "A1": None,   # legacy corpus x legacydict run  (also source of legacy staged chunks + vocab)
    "A2": None,   # BERTopic run (only its Model_finetuning/trained_encoder is used;
                  #  A2 shares A1's CP1/CP2 staged chunks + vocabulary)
    "B1": None,   # policy corpus x policydict run  (also source of policy staged chunks + vocab)
}

# Which run folder supplies each corpus's staged chunks (CP1) + vocabulary (CP2).
CORPUS_FROM_RUN = {"legacy": "A1", "policy": "B1"}

# --- direct overrides (optional) ------------------------------------------
# Set any of these to a file/folder path to bypass run-folder resolution.
OVERRIDES = {
    "chunks_legacy": None,      # chunked_corpus.csv of the legacy corpus
    "chunks_policy": None,      # chunked_corpus.csv of the (Tier-1-filtered) policy corpus
    "vocab_legacy": None,       # vocabulary.csv of the legacy corpus
    "vocab_policy": None,       # vocabulary.csv of the policy corpus
    "dict_legacydict": None,    # curated legacy dictionary csv
    "dict_policydict": None,    # curated (pruned) policy dictionary csv
    "encoder_A1": None,         # Model_finetuning/trained_encoder of A1
    "encoder_A2": None,
    "encoder_B1": None,
}

# --- models ----------------------------------------------------------------
# Internal model codes only. A finetuned model scores ONLY with the dictionary
# it learned from (no cross-dictionary combinations).
MODELS = {
    "A1": {"dict": "legacydict", "trained_on": "legacy"},
    "A2": {"dict": "legacydict", "trained_on": "legacy"},
    "B1": {"dict": "policydict", "trained_on": "policy"},
}
CORPORA = ["legacy", "policy"]

# Pretrained-e5 baseline (mean-centred baselines for Ch2/Ch3/Ch4).
# The pretrained model has learned no dictionary, so it may score with BOTH
# dictionaries without violating the cross-dictionary rule.
INCLUDE_PRETRAINED_BASELINE = True
PRETRAINED_LABEL = "pre"                  # internal code; combined tables label it "pretrained"
PRETRAINED_MODEL_NAME = "intfloat/multilingual-e5-base"

# --- combined per-corpus chunk tables --------------------------------------
# Column order: chunk_index, Raw_text, Doc_type, Year, notes, Region, then one
# score block per model in MODEL_BLOCK_ORDER (defined in the combined-table cell):
# pretrained, A1, A2, B1.
# `notes` flags chunks found in these files (matched on the stable content key;
# a file without a content_key column is matched on its text column instead).
NOTES_SOURCES = {
    "used in prose": None,          # e.g. PROJECT_ROOT / "workflow_43" / "prose_chunks.csv"
    "in previous dataset": None,    # e.g. PROJECT_ROOT / "workflow_43" / "legacy_chunk_scores_v2.parquet"
}
COMBINED_XLSX_MAX_ROWS = 20000      # also write .xlsx when the corpus has at most this many chunks

# --- encoding --------------------------------------------------------------
# The finetunes are based on intfloat/multilingual-e5-base. Training encoded
# raw strings (no "query:"/"passage:" prefixes), so scoring must match.
# Only set these if the training notebooks used prefixes.
E5_CHUNK_PREFIX = ""
E5_TERM_PREFIX = ""

MAX_SEQ_LENGTH = 512          # e5 supports 512; fixes the old 128-token truncation
CHUNK_BATCH_SIZE = 128
TERM_BATCH_SIZE = 512

# --- scoring ---------------------------------------------------------------
APPLY_MEAN_CENTERING = True   # per-space embedding mean (see notebook header)

# --- run behaviour ---------------------------------------------------------
RESUME = True                 # skip embeddings/scores whose outputs already exist
EXPECTED_CHUNKS = {"legacy": 7943, "policy": 186170}   # warn (not fail) on mismatch

RUN_STAMP = datetime.now().strftime("%Y%m%d")
OUTPUT_DIR = TRAIN_WORKFLOW_DIR / f"full_embed_score_e5_{RUN_STAMP}"

print(f"PROJECT_ROOT       : {PROJECT_ROOT}")
print(f"TRAIN_WORKFLOW_DIR : {TRAIN_WORKFLOW_DIR}  (exists: {TRAIN_WORKFLOW_DIR.exists()})")
print(f"OUTPUT_DIR         : {OUTPUT_DIR}")

---
## Helpers

Case-insensitive product lookup, robust CSV loaders (column names drifted across
pipeline versions), stable content keys, and linear-algebra utilities.


In [ ]:
# ============================================================
# CELL 3: HELPERS
# ============================================================

def find_file(folder, *names, patterns=()):
    """Case-insensitive lookup of the first existing file in `folder`.

    `names` are exact filenames tried case-insensitively; `patterns` are
    lowercase glob patterns tried afterwards (newest match wins)."""
    folder = Path(folder)
    if not folder.exists():
        return None
    listing = {p.name.lower(): p for p in folder.iterdir() if p.is_file()}
    for name in names:
        hit = listing.get(name.lower())
        if hit is not None:
            return hit
    for pattern in patterns:
        hits = [p for p in folder.iterdir()
                if p.is_file() and re.fullmatch(pattern, p.name.lower())]
        if hits:
            return max(hits, key=lambda p: p.stat().st_mtime)
    return None


def normalize_text(text):
    """Normalization used for the stable content key (NFKC, casefold, collapsed whitespace)."""
    text = unicodedata.normalize("NFKC", str(text))
    return re.sub(r"\s+", " ", text).strip().lower()


def content_key(text):
    """SHA-1 of the normalized chunk text - stable across chunk-boundary shifts."""
    return hashlib.sha1(normalize_text(text).encode("utf-8")).hexdigest()


def pick_col(df, options, required=True, what=""):
    lower = {c.lower(): c for c in df.columns}
    for opt in options:
        if opt in lower:
            return lower[opt]
    if required:
        raise KeyError(f"Could not find a {what} column among {list(df.columns)} "
                       f"(tried {options})")
    return None


def load_chunked_corpus(path, corpus_label):
    """Load a CP1 chunked_corpus.csv into a standardized frame with content keys."""
    df = pd.read_csv(path)
    text_col = pick_col(df, ["chunk_text", "text", "chunk", "content", "clean_text"], what="chunk text")
    doc_col = pick_col(df, ["doc_id", "document", "filename", "source_file", "file", "doc"],
                       required=False, what="doc id")
    idx_col = pick_col(df, ["chunk_index", "chunk_idx", "chunk_id", "chunk_nr"],
                       required=False, what="chunk index")

    out = pd.DataFrame({"text": df[text_col].astype(str)})
    out["doc_id"] = df[doc_col].astype(str) if doc_col else ""
    out["chunk_index"] = df[idx_col] if idx_col else np.arange(len(df))
    for extra, target in (("year", "year"), ("doc_type", "doc_type"),
                          ("doctype", "doc_type"), ("region", "region"),
                          ("regio", "region")):
        col = pick_col(df, [extra], required=False)
        if col and target not in out.columns:
            out[target] = df[col]
    out["content_key"] = [content_key(t) for t in out["text"]]

    dupes = out["content_key"].duplicated().sum()
    print(f"  [{corpus_label}] {len(out):,} chunks from {path}")
    if dupes:
        print(f"  [{corpus_label}] note: {dupes:,} duplicate content keys (identical chunk texts)")
    expected = EXPECTED_CHUNKS.get(corpus_label)
    if expected is not None and len(out) != expected:
        print(f"  [{corpus_label}] WARNING: expected {expected:,} chunks, found {len(out):,} - "
              f"check the staged corpus (Tier-1 doc_type filter for policy).")
    return out


def load_vocabulary(path, corpus_label):
    """Load a CP2 vocabulary (uncapped: unigrams + surviving n-grams)."""
    path = Path(path)
    if path.suffix.lower() == ".json":
        with open(path, encoding="utf-8") as fh:
            data = json.load(fh)
        if isinstance(data, dict):
            df = pd.DataFrame({"term": list(data.keys()), "frequency": list(data.values())})
        else:
            df = pd.DataFrame({"term": list(data)})
    else:
        df = pd.read_csv(path)
    term_col = pick_col(df, ["term", "word", "token", "ngram", "vocab"], what="term")
    freq_col = pick_col(df, ["frequency", "freq", "count", "tf", "term_freq"], required=False)
    type_col = pick_col(df, ["term_type", "type", "kind"], required=False)

    out = pd.DataFrame({"term": df[term_col].astype(str)})
    out["frequency"] = df[freq_col] if freq_col else np.nan
    if type_col:
        out["term_type"] = df[type_col].astype(str)
    else:
        out["term_type"] = np.where(out["term"].str.contains(" "), "ngram", "unigram")
    out = out.drop_duplicates(subset="term").reset_index(drop=True)
    n_uni = (out["term_type"] == "unigram").sum()
    print(f"  [{corpus_label}] vocabulary: {len(out):,} terms "
          f"({n_uni:,} unigrams, {len(out) - n_uni:,} n-grams) from {path}")
    return out


def load_dictionary(path, dict_label):
    """Load a curated dictionary csv -> (topic, seed) rows."""
    df = pd.read_csv(path)
    topic_col = pick_col(df, ["topic", "theme", "category", "label"], what="topic")
    seed_col = pick_col(df, ["keyword", "term", "seed", "word"], what="seed/keyword")
    out = pd.DataFrame({
        "topic": df[topic_col].astype(str).str.strip(),
        "seed": df[seed_col].astype(str).str.strip(),
    })
    weight_col = pick_col(df, ["weight", "score"], required=False)
    out["weight"] = df[weight_col] if weight_col else 1.0
    out = out[(out["seed"] != "") & (out["topic"] != "")]
    out = out.drop_duplicates(subset=["topic", "seed"]).reset_index(drop=True)
    print(f"  [{dict_label}] {out['seed'].nunique():,} unique seeds across "
          f"{out['topic'].nunique()} topics from {path}")
    return out


def l2norm(mat, eps=1e-12):
    return mat / np.maximum(np.linalg.norm(mat, axis=1, keepdims=True), eps)


def slug(text):
    return re.sub(r"[^0-9a-zA-Z]+", "_", str(text)).strip("_").lower()

print("helpers defined")

---
## Discovery & validation

Resolves the run folders and products, then builds the job list. Models whose
`trained_encoder` is missing (e.g. **A1 still training**) are skipped with a warning —
re-run the notebook later to fill them in.


In [ ]:
# ============================================================
# CELL 4: DISCOVERY & JOB LIST
# ============================================================

def discover_run_folders(base):
    """List candidate pipeline run folders (any folder holding the standard layout)."""
    base = Path(base)
    candidates = []
    search_roots = [base] if base.exists() else []
    if not search_roots:  # fall back to scanning all workflow_* dirs under the project root
        search_roots = sorted(PROJECT_ROOT.glob("workflow_*"))
    for root in search_roots:
        for sub in sorted(root.iterdir()):
            if not sub.is_dir():
                continue
            markers = {
                "trained_encoder": (sub / "Model_finetuning" / "trained_encoder").exists(),
                "chunked_corpus": (sub / "Other_data" / "chunked_corpus.csv").exists(),
                "vocabulary": find_file(sub / "Other_data", "vocabulary.csv", "vocabulary.json") is not None,
                "dictionary": find_file(sub / "Dictionary", "Curated_dictionary.csv",
                                        "curated_dictionary.csv") is not None,
            }
            if any(markers.values()):
                candidates.append((sub, markers))
    return candidates

candidates = discover_run_folders(TRAIN_WORKFLOW_DIR)
print(f"Candidate run folders under {TRAIN_WORKFLOW_DIR}:")
if not candidates:
    print("  (none found - check TRAIN_WORKFLOW_DIR or set OVERRIDES directly)")
for sub, markers in candidates:
    flags = " ".join(k for k, v in markers.items() if v)
    print(f"  - {sub.name:60s} [{flags}]")

def _guess_run_folder(label):
    """Best-effort auto-assignment from folder names; None when ambiguous."""
    name_hints = {
        "A1": ("legacy", "slav"), "A2": ("bertopic",), "B1": ("policy",),
    }[label]
    hits = [sub for sub, markers in candidates
            if any(h in sub.name.lower() for h in name_hints)]
    if label != "A2":  # A2's folder is the bertopic one; exclude it from A1/B1 guesses
        hits = [h for h in hits if "bertopic" not in h.name.lower()]
    return hits[0] if len(hits) == 1 else None

resolved_runs = {}
for label in MODELS:
    folder = RUN_FOLDERS.get(label)
    if folder is None:
        folder = _guess_run_folder(label)
        if folder is not None:
            print(f"auto-assigned RUN_FOLDERS['{label}'] = {folder}")
    if folder is not None:
        folder = Path(folder)
    resolved_runs[label] = folder

def _resolve(key, run_label, *relnames, subfolder="Other_data", patterns=()):
    if OVERRIDES.get(key):
        return Path(OVERRIDES[key])
    run = resolved_runs.get(run_label)
    if run is None:
        return None
    return find_file(run / subfolder, *relnames, patterns=patterns)

RESOLVED = {}
for corpus in CORPORA:
    src = CORPUS_FROM_RUN[corpus]
    RESOLVED[f"chunks_{corpus}"] = _resolve(f"chunks_{corpus}", src, "chunked_corpus.csv")
    RESOLVED[f"vocab_{corpus}"] = _resolve(f"vocab_{corpus}", src, "vocabulary.csv", "vocabulary.json")

RESOLVED["dict_legacydict"] = _resolve("dict_legacydict", "A1",
                                       "Curated_dictionary.csv", "curated_dictionary.csv",
                                       subfolder="Dictionary")
RESOLVED["dict_policydict"] = _resolve("dict_policydict", "B1",
                                       "Curated_dictionary.csv", "curated_dictionary.csv",
                                       subfolder="Dictionary")

for label in MODELS:
    if OVERRIDES.get(f"encoder_{label}"):
        RESOLVED[f"encoder_{label}"] = Path(OVERRIDES[f"encoder_{label}"])
    elif resolved_runs.get(label) is not None:
        enc = resolved_runs[label] / "Model_finetuning" / "trained_encoder"
        RESOLVED[f"encoder_{label}"] = enc if enc.exists() else None
    else:
        RESOLVED[f"encoder_{label}"] = None

print("\nResolved products:")
missing = []
for key, val in RESOLVED.items():
    status = "OK " if val is not None else "-- "
    print(f"  {status} {key:18s} -> {val}")
    if val is None:
        missing.append(key)

# --- build the job list ----------------------------------------------------
active_models = {}
for label, spec in MODELS.items():
    if RESOLVED.get(f"encoder_{label}") is None:
        print(f"\nWARNING: encoder for {label} not found - {label} is skipped this run "
              f"(still training?). Re-run the notebook once it lands; RESUME keeps finished products.")
        continue
    active_models[label] = {**spec, "encoder_path": str(RESOLVED[f"encoder_{label}"])}

if INCLUDE_PRETRAINED_BASELINE:
    # pretrained baseline scores with BOTH dictionaries (allowed: it learned neither)
    active_models[PRETRAINED_LABEL] = {"dict": "both", "trained_on": None,
                                       "encoder_path": PRETRAINED_MODEL_NAME}

JOBS = []  # (model_label, corpus, dict_label)
for label, spec in active_models.items():
    dicts = ["legacydict", "policydict"] if spec["dict"] == "both" else [spec["dict"]]
    for corpus in CORPORA:
        for dict_label in dicts:
            JOBS.append((label, corpus, dict_label))

needed_data = ({f"chunks_{c}" for _, c, _ in JOBS} | {f"vocab_{c}" for _, c, _ in JOBS}
               | {f"dict_{d}" for _, _, d in JOBS})
blocking = [k for k in needed_data if RESOLVED.get(k) is None]
if blocking:
    raise FileNotFoundError(
        f"Missing required products for the current job list: {blocking}. "
        f"Set RUN_FOLDERS or OVERRIDES in Cell 2 and re-run from Cell 4.")

print(f"\nJob list ({len(JOBS)} scoring jobs):")
for label, corpus, dict_label in JOBS:
    print(f"  {label:4s} x {corpus:6s} corpus+vocab  (dict: {dict_label})")

---
## Load staged data

Chunked corpora (CP1), vocabularies (CP2, uncapped) and curated dictionaries.
Also reports seed→vocab coverage per corpus as a sanity check.


In [ ]:
# ============================================================
# CELL 5: LOAD CORPORA, VOCABULARIES, DICTIONARIES
# ============================================================
print("Loading staged chunks:")
chunks = {c: load_chunked_corpus(RESOLVED[f"chunks_{c}"], c)
          for c in sorted({c for _, c, _ in JOBS})}

print("\nLoading vocabularies:")
vocabs = {c: load_vocabulary(RESOLVED[f"vocab_{c}"], c)
          for c in sorted({c for _, c, _ in JOBS})}

print("\nLoading dictionaries:")
dictionaries = {d: load_dictionary(RESOLVED[f"dict_{d}"], d)
                for d in sorted({d for _, _, d in JOBS})}

print("\nSeed coverage in corpus vocabularies (informational):")
for dict_label, ddf in dictionaries.items():
    seeds = set(ddf["seed"].str.lower())
    for corpus, vdf in vocabs.items():
        terms = set(vdf["term"].str.lower())
        hit = len(seeds & terms)
        print(f"  {dict_label:11s} in {corpus:6s} vocab: {hit}/{len(seeds)} seeds present")

---
## Embedding

One encoder in memory at a time. Per model: seed terms of its dictionary, then per
corpus the chunks and the vocabulary terms. Raw float32 embeddings go to `.npy`
(row order matches the saved `index.parquet`); per-space means are computed and
saved in the scoring cell. `RESUME = True` skips arrays that already exist with
the right shape.


In [ ]:
# ============================================================
# CELL 6: EMBED CHUNKS, VOCAB TERMS AND SEEDS
# ============================================================
for sub in ("config", "reports", "Chunk_scores", "Vocab_scores",
            "embeddings/means"):
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

def _existing_ok(path, n_rows):
    if not (RESUME and Path(path).exists()):
        return False
    try:
        arr = np.load(path, mmap_mode="r")
        return arr.shape[0] == n_rows
    except Exception:
        return False

def embed_texts(encoder, texts, batch_size, prefix, desc):
    if prefix:
        texts = [prefix + t for t in texts]
    out = np.empty((len(texts), encoder.get_sentence_embedding_dimension()), dtype=np.float32)
    for start in tqdm(range(0, len(texts), batch_size), desc=desc, leave=False):
        batch = texts[start:start + batch_size]
        emb = encoder.encode(batch, batch_size=batch_size, show_progress_bar=False,
                             normalize_embeddings=False, convert_to_numpy=True)
        out[start:start + len(batch)] = emb.astype(np.float32)
    return out

def _write_meta(npy_path, model_label, encoder_path, kind, corpus, n_rows, dim, source):
    meta = {
        "model": model_label, "encoder_path": str(encoder_path), "kind": kind,
        "corpus": corpus, "rows": int(n_rows), "dim": int(dim),
        "chunk_prefix": E5_CHUNK_PREFIX, "term_prefix": E5_TERM_PREFIX,
        "max_seq_length": MAX_SEQ_LENGTH, "normalize_at_encode": False,
        "centering": "none (raw embeddings; per-space means saved separately)",
        "source": str(source), "created": datetime.now().isoformat(timespec="seconds"),
    }
    with open(str(npy_path).replace(".npy", "__meta.json"), "w", encoding="utf-8") as fh:
        json.dump(meta, fh, indent=2)

emb_index_written = set()
model_order = list(dict.fromkeys(label for label, _, _ in JOBS))

for model_label in model_order:
    spec = active_models[model_label]
    encoder = None

    def get_encoder():
        # lazy-load so fully resumed models never touch the GPU
        global encoder
        if encoder is None:
            print(f"  loading encoder for {model_label}: {spec['encoder_path']}")
            encoder = SentenceTransformer(spec["encoder_path"], device=DEVICE)
            encoder.max_seq_length = MAX_SEQ_LENGTH
        return encoder

    print(f"\n=== {model_label} ===")

    # --- seed embeddings (per dictionary this model scores with) ---
    dicts = ["legacydict", "policydict"] if spec["dict"] == "both" else [spec["dict"]]
    for dict_label in dicts:
        seed_dir = OUTPUT_DIR / "embeddings" / "seeds" / dict_label
        seed_dir.mkdir(parents=True, exist_ok=True)
        ddf = dictionaries[dict_label]
        seed_terms = ddf["seed"].drop_duplicates().tolist()
        npy_path = seed_dir / f"{model_label}.npy"
        if _existing_ok(npy_path, len(seed_terms)):
            print(f"  seeds/{dict_label}/{model_label}.npy exists - skipped")
        else:
            emb = embed_texts(get_encoder(), seed_terms, TERM_BATCH_SIZE, E5_TERM_PREFIX,
                              f"{model_label} seeds {dict_label}")
            np.save(npy_path, emb)
            pd.DataFrame({"seed": seed_terms}).to_parquet(seed_dir / f"{model_label}__index.parquet")
            _write_meta(npy_path, model_label, spec["encoder_path"], "seeds", dict_label,
                        len(seed_terms), emb.shape[1], RESOLVED[f"dict_{dict_label}"])

    # --- chunk + vocab embeddings per corpus ---
    for corpus in CORPORA:
        chunk_dir = OUTPUT_DIR / "embeddings" / "chunks" / corpus
        vocab_dir = OUTPUT_DIR / "embeddings" / "vocab" / corpus
        chunk_dir.mkdir(parents=True, exist_ok=True)
        vocab_dir.mkdir(parents=True, exist_ok=True)

        if corpus not in emb_index_written:
            chunks[corpus][["content_key", "doc_id", "chunk_index"]].to_parquet(
                chunk_dir / "index.parquet")
            vocabs[corpus][["term", "term_type", "frequency"]].to_parquet(
                vocab_dir / "index.parquet")
            emb_index_written.add(corpus)

        npy_path = chunk_dir / f"{model_label}.npy"
        if _existing_ok(npy_path, len(chunks[corpus])):
            print(f"  chunks/{corpus}/{model_label}.npy exists - skipped")
        else:
            emb = embed_texts(get_encoder(), chunks[corpus]["text"].tolist(),
                              CHUNK_BATCH_SIZE, E5_CHUNK_PREFIX,
                              f"{model_label} chunks {corpus}")
            np.save(npy_path, emb)
            _write_meta(npy_path, model_label, spec["encoder_path"], "chunks", corpus,
                        emb.shape[0], emb.shape[1], RESOLVED[f"chunks_{corpus}"])
            print(f"  saved chunks/{corpus}/{model_label}.npy {emb.shape}")

        npy_path = vocab_dir / f"{model_label}.npy"
        if _existing_ok(npy_path, len(vocabs[corpus])):
            print(f"  vocab/{corpus}/{model_label}.npy exists - skipped")
        else:
            emb = embed_texts(get_encoder(), vocabs[corpus]["term"].tolist(),
                              TERM_BATCH_SIZE, E5_TERM_PREFIX,
                              f"{model_label} vocab {corpus}")
            np.save(npy_path, emb)
            _write_meta(npy_path, model_label, spec["encoder_path"], "vocab", corpus,
                        emb.shape[0], emb.shape[1], RESOLVED[f"vocab_{corpus}"])
            print(f"  saved vocab/{corpus}/{model_label}.npy {emb.shape}")

    if encoder is not None:
        del encoder
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

print("\nembedding stage complete")

---
## Scoring

Per job (model × corpus × dictionary):

1. load the raw embeddings,
2. compute the **per-space mean** (chunk space and term space separately, per
   corpus × model) and save it,
3. centre embeddings *and* seed embeddings with the mean of the space being scored,
4. build unit-norm topic centroids from the centred seeds,
5. cosine-score chunks and vocab terms; write parquets with `cos__<topic>` columns,
   `top_topic`, `top_score` and the top1–top2 `margin`.

The vocab-term space also scores each dictionary seed itself (`is_seed = True` where a
vocab term is a seed).


In [ ]:
# ============================================================
# CELL 7: MEAN-CENTER + SCORE  (chunks and vocab terms)
# ============================================================

def build_centroids(seed_emb_raw, seed_terms, ddf, space_mean):
    """Unit-norm topic centroids from seed embeddings centred with the target space mean."""
    centred = seed_emb_raw - space_mean if APPLY_MEAN_CENTERING else seed_emb_raw
    centred = l2norm(centred)
    pos = {t: i for i, t in enumerate(seed_terms)}
    topics = sorted(ddf["topic"].unique())
    centroids = np.zeros((len(topics), centred.shape[1]), dtype=np.float32)
    for k, topic in enumerate(topics):
        rows = [pos[s] for s in ddf.loc[ddf["topic"] == topic, "seed"] if s in pos]
        centroids[k] = centred[rows].mean(axis=0)
    return topics, l2norm(centroids)

def score_space(emb_raw, space_mean, centroids):
    centred = emb_raw - space_mean if APPLY_MEAN_CENTERING else emb_raw
    return l2norm(centred) @ centroids.T   # cosine, rows x topics

def attach_scores(base_df, scores, topics):
    out = base_df.copy()
    for k, topic in enumerate(topics):
        out[f"cos__{slug(topic)}"] = scores[:, k]
    top_idx = scores.argmax(axis=1)
    out["top_topic"] = [topics[i] for i in top_idx]
    out["top_score"] = scores.max(axis=1)
    if scores.shape[1] > 1:
        part = np.partition(scores, -2, axis=1)
        out["margin"] = part[:, -1] - part[:, -2]
    else:
        out["margin"] = np.nan
    return out

spread_rows = []
means_dir = OUTPUT_DIR / "embeddings" / "means"

for model_label, corpus, dict_label in JOBS:
    chunk_out = (OUTPUT_DIR / "Chunk_scores" /
                 f"chunk_scores_{corpus}__dict-{dict_label}__model-{model_label}.parquet")
    vocab_out = (OUTPUT_DIR / "Vocab_scores" /
                 f"vocab_scores_{corpus}__dict-{dict_label}__model-{model_label}.parquet")
    if RESUME and chunk_out.exists() and vocab_out.exists():
        print(f"{model_label} x {corpus} x {dict_label}: score parquets exist - skipped")
        continue

    chunk_emb = np.load(OUTPUT_DIR / "embeddings" / "chunks" / corpus / f"{model_label}.npy")
    vocab_emb = np.load(OUTPUT_DIR / "embeddings" / "vocab" / corpus / f"{model_label}.npy")
    seed_dir = OUTPUT_DIR / "embeddings" / "seeds" / dict_label
    seed_emb = np.load(seed_dir / f"{model_label}.npy")
    seed_terms = pd.read_parquet(seed_dir / f"{model_label}__index.parquet")["seed"].tolist()
    ddf = dictionaries[dict_label]

    chunk_mean = chunk_emb.mean(axis=0, keepdims=True)
    vocab_mean = vocab_emb.mean(axis=0, keepdims=True)
    np.save(means_dir / f"chunks__{corpus}__{model_label}.npy", chunk_mean)
    np.save(means_dir / f"vocab__{corpus}__{model_label}.npy", vocab_mean)

    # --- chunk scoring (chunk-space mean for chunks AND centroids) ---
    topics, centroids_c = build_centroids(seed_emb, seed_terms, ddf, chunk_mean)
    chunk_scores = score_space(chunk_emb, chunk_mean, centroids_c)
    cdf = attach_scores(chunks[corpus][["content_key", "doc_id", "chunk_index"]],
                        chunk_scores, topics)
    cdf.to_parquet(chunk_out, index=False)

    # --- vocab scoring (term-space mean for terms AND centroids) ---
    topics_v, centroids_v = build_centroids(seed_emb, seed_terms, ddf, vocab_mean)
    vocab_scores = score_space(vocab_emb, vocab_mean, centroids_v)
    vdf = attach_scores(vocabs[corpus][["term", "term_type", "frequency"]],
                        vocab_scores, topics_v)
    seeds_lower = set(ddf["seed"].str.lower())
    vdf["is_seed"] = vdf["term"].str.lower().isin(seeds_lower)
    vdf.to_parquet(vocab_out, index=False)

    for space, scores in (("chunks", chunk_scores), ("vocab", vocab_scores)):
        margins = (np.partition(scores, -2, axis=1)[:, -1]
                   - np.partition(scores, -2, axis=1)[:, -2]) if scores.shape[1] > 1 else np.array([np.nan])
        spread_rows.append({
            "model": model_label, "corpus": corpus, "dict": dict_label, "space": space,
            "rows": scores.shape[0], "topics": scores.shape[1],
            "top_mean": float(scores.max(axis=1).mean()),
            "top_std": float(scores.max(axis=1).std()),
            "top_min": float(scores.max(axis=1).min()),
            "top_max": float(scores.max(axis=1).max()),
            "margin_mean": float(np.nanmean(margins)),
        })
    print(f"{model_label} x {corpus} x {dict_label}: "
          f"chunks {chunk_scores.shape} -> {chunk_out.name} | "
          f"vocab {vocab_scores.shape} -> {vocab_out.name}")

print("\nscoring stage complete")

---
## Combined per-corpus chunk tables

One wide table per corpus, one row per chunk. Fixed leading columns —
`chunk_index, Raw_text, Doc_type, Year, notes, Region` — then a score block per
model in the order **pretrained, A1, A2, B1** (per-topic cosines + `top_topic` +
`top_score`; the pretrained blocks come once per dictionary). `doc_id` and
`content_key` close the table. Missing models (e.g. A1 still training) simply
have no block yet — re-running this cell after a resume rebuilds the tables.


In [ ]:
# ============================================================
# CELL 8: COMBINED PER-CORPUS CHUNK TABLES
# ============================================================
MODEL_BLOCK_ORDER = [PRETRAINED_LABEL, "A1", "A2", "B1"]
DISPLAY_LABEL = {PRETRAINED_LABEL: "pretrained"}

def _load_notes_lookup():
    lookup = {}
    for label, path in NOTES_SOURCES.items():
        if not path:
            continue
        path = Path(path)
        if not path.exists():
            print(f"  notes source '{label}' not found: {path} - skipped")
            continue
        suffix = path.suffix.lower()
        if suffix in (".xlsx", ".xls"):
            src = pd.read_excel(path)
        elif suffix == ".parquet":
            src = pd.read_parquet(path)
        else:
            src = pd.read_csv(path)
        key_col = pick_col(src, ["content_key"], required=False)
        if key_col:
            keys = src[key_col].astype(str)
        else:
            text_col = pick_col(src, ["raw_text", "chunk_text", "text", "chunk", "content"],
                                required=False, what="text")
            if text_col is None:
                print(f"  notes source '{label}': no content_key or text column - skipped")
                continue
            keys = src[text_col].map(content_key)
        for key in keys:
            lookup.setdefault(key, [])
            if label not in lookup[key]:
                lookup[key].append(label)
        print(f"  notes source '{label}': {keys.nunique():,} keys from {path.name}")
    return lookup

print("Notes sources:")
notes_lookup = _load_notes_lookup()
if not notes_lookup:
    print("  (none configured - `notes` column will be empty)")

for corpus in CORPORA:
    base = chunks[corpus]
    combined = pd.DataFrame({
        "chunk_index": base["chunk_index"].values,
        "Raw_text": base["text"].values,
        "Doc_type": base["doc_type"].values if "doc_type" in base.columns else "",
        "Year": base["year"].values if "year" in base.columns else "",
        "notes": ["; ".join(notes_lookup.get(k, [])) for k in base["content_key"]],
        "Region": base["region"].values if "region" in base.columns else "",
    })

    blocks_added = []
    for model_label in MODEL_BLOCK_ORDER:
        if model_label not in active_models:
            continue
        spec = active_models[model_label]
        disp = DISPLAY_LABEL.get(model_label, model_label)
        dicts = ["legacydict", "policydict"] if spec["dict"] == "both" else [spec["dict"]]
        multi_dict = len(dicts) > 1
        for dict_label in dicts:
            score_path = (OUTPUT_DIR / "Chunk_scores" /
                          f"chunk_scores_{corpus}__dict-{dict_label}__model-{model_label}.parquet")
            if not score_path.exists():
                print(f"  [{corpus}] {score_path.name} missing - block skipped")
                continue
            sdf = pd.read_parquet(score_path)
            assert len(sdf) == len(combined), f"row-count mismatch for {score_path.name}"
            assert (sdf["content_key"].values == base["content_key"].values).all(), \
                f"row-order mismatch for {score_path.name}"
            prefix = f"{disp}__{dict_label}" if multi_dict else disp
            for col in sdf.columns:
                if col.startswith("cos__"):
                    combined[f"{prefix}__{col[len('cos__'):]}"] = sdf[col].values
            combined[f"{prefix}__top_topic"] = sdf["top_topic"].values
            combined[f"{prefix}__top_score"] = sdf["top_score"].values
            blocks_added.append(prefix)

    combined["doc_id"] = base["doc_id"].values
    combined["content_key"] = base["content_key"].values

    out_parquet = OUTPUT_DIR / "Chunk_scores" / f"chunk_scores_combined_{corpus}.parquet"
    combined.to_parquet(out_parquet, index=False)
    written = out_parquet.name
    if len(combined) <= COMBINED_XLSX_MAX_ROWS:
        try:
            xlsx_path = out_parquet.with_suffix(".xlsx")
            combined.to_excel(xlsx_path, index=False)
            written += f" + {xlsx_path.name}"
        except Exception as exc:
            print(f"  [{corpus}] xlsx write skipped: {exc}")
    print(f"[{corpus}] combined table {combined.shape} -> {written}")
    print(f"         blocks: {', '.join(blocks_added) if blocks_added else '(none)'}")

---
## Report & run manifest

Score-spread summary per space (the check that mean-centering actually opened up the
e5 score distribution — a near-zero `top_std` / `margin_mean` means a flat space) and a
config snapshot with every resolved path, so the run is reproducible.

**Reminder:** compare rows only *within* a corpus. Legacy-space and policy-space scores
are centred with different means and are not comparable in absolute terms.


In [ ]:
# ============================================================
# CELL 9: SPREAD REPORT + MANIFEST
# ============================================================
if spread_rows:
    spread_df = pd.DataFrame(spread_rows)
    report_path = OUTPUT_DIR / "reports" / "score_spread_summary.csv"
    if RESUME and report_path.exists():
        prev = pd.read_csv(report_path)
        spread_df = (pd.concat([prev, spread_df])
                     .drop_duplicates(subset=["model", "corpus", "dict", "space"], keep="last"))
    spread_df = spread_df.sort_values(["corpus", "space", "model"]).reset_index(drop=True)
    spread_df.to_csv(report_path, index=False)
    display(spread_df)

    flat = spread_df[(spread_df["top_std"] < 0.02) | (spread_df["margin_mean"] < 0.005)]
    if len(flat):
        print("\nWARNING: near-flat score spaces detected (check centering / encoder):")
        for _, row in flat.iterrows():
            print(f"  {row['model']} x {row['corpus']} x {row['space']}: "
                  f"top_std={row['top_std']:.4f} margin_mean={row['margin_mean']:.4f}")
else:
    print("no new scoring jobs ran (everything resumed)")

manifest = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "notebook": "Embed_score_corpus_vocab_v1.ipynb",
    "device": DEVICE,
    "train_workflow_dir": str(TRAIN_WORKFLOW_DIR),
    "output_dir": str(OUTPUT_DIR),
    "resolved_products": {k: str(v) for k, v in RESOLVED.items()},
    "run_folders": {k: str(v) for k, v in resolved_runs.items()},
    "active_models": active_models,
    "skipped_models": [m for m in MODELS if m not in active_models],
    "jobs": [{"model": m, "corpus": c, "dict": d} for m, c, d in JOBS],
    "settings": {
        "apply_mean_centering": APPLY_MEAN_CENTERING,
        "centering_rule": "per-space mean (chunk space and term space separately, per corpus x model)",
        "max_seq_length": MAX_SEQ_LENGTH,
        "e5_chunk_prefix": E5_CHUNK_PREFIX,
        "e5_term_prefix": E5_TERM_PREFIX,
        "include_pretrained_baseline": INCLUDE_PRETRAINED_BASELINE,
        "expected_chunks": EXPECTED_CHUNKS,
        "notes_sources": {k: str(v) if v else None for k, v in NOTES_SOURCES.items()},
        "combined_table_block_order": ["pretrained", "A1", "A2", "B1"],
    },
    "combined_tables": [f"Chunk_scores/chunk_scores_combined_{c}.parquet" for c in CORPORA],
    "counts": {
        "chunks": {c: int(len(df)) for c, df in chunks.items()},
        "vocab_terms": {c: int(len(df)) for c, df in vocabs.items()},
        "seeds": {d: int(df["seed"].nunique()) for d, df in dictionaries.items()},
    },
}
manifest_path = OUTPUT_DIR / "config" / f"run_config_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(manifest_path, "w", encoding="utf-8") as fh:
    json.dump(manifest, fh, indent=2, ensure_ascii=False)
print(f"\nmanifest -> {manifest_path}")

skipped = [m for m in MODELS if m not in active_models]
if skipped:
    print(f"\nNOTE: skipped models this run: {skipped}. Re-run the notebook once their "
          f"trained_encoder exists; RESUME will only compute the missing products.")
print("run complete")